# Marketing Campaign Effectiveness Analysis

**Author:** Suhaiv Man Singh

**Goal:** find out which campaigns, campaign types, timing and customer groups get the best response, using the *Complete Journey* retail dataset.

Notebook outline:

0. Dataset source and background
1. Step 1: Load and explore the data
2. Step 2: Clean and transform the data (missing values, outliers, encoding, new features)
3. Step 3: Exploratory data analysis (EDA) and key insights

## 0. Dataset source and background

| | |
|---|---|
| **Name** | The Complete Journey (retail shopping data from the 84.51° Complete Journey study) |
| **Stored in this project** | `Dataset/raw/` as CSV files (exported once from the Python package `completejourney_py` by `download_data.py`) |
| **What it contains** | One year (2017) of grocery transactions for 2,469 households, plus 27 coupon campaigns, which households each campaign targeted, and which coupons were redeemed |

**Note on the data source:** the R version of this dataset credits 84.51° and describes the households as frequent shoppers at a retailer. The README of the Python package calls the data "simulated data based on real grocery shopping patterns" meant for educational use. This project is a learning exercise, so results describe this dataset only and should not be treated as findings about a real retailer.

**How the task maps to this dataset.** The task lists metrics such as click-through rate, cost per acquisition, channel and region as *examples*. This dataset has no impressions, clicks or region, so these are replaced by close equivalents:

| Task example | Used here |
|---|---|
| Response rate | **Redemption rate** = share of targeted households that redeemed at least one coupon |
| Cost per acquisition | **Cost per redemption** = coupon discount given out / number of redemptions |
| ROI | **Sales per discount dollar** = sales of targeted households / coupon discount given (a simple proxy, not true incremental return) |
| Engagement | **Baskets per household** (shopping trips) |
| Channel | **Campaign type** (Type A, B or C) |
| Target audience | Household **age, income, household composition** |
| Timing | **Start month** and **duration** of the campaign |

**Tables used**

| Table | One row is | Used for |
|---|---|---|
| `campaign_descriptions` | a campaign | type, start and end dates |
| `campaigns` | a household targeted by a campaign | who received which campaign |
| `coupon_redemptions` | a redeemed coupon | the response |
| `coupons` | a coupon offered in a campaign | number of coupons offered |
| `transactions` | a purchased product line | sales, discounts, baskets |
| `demographics` | a household | age, income, household make-up |

# Step 1: Load and explore the data

## 1.1 Load the data

In [1]:
# Import the libraries used in this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Show every column when printing tables, and use clean plot defaults
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [2]:
campaign_desc = pd.read_csv("Dataset/campaign_descriptions.csv", parse_dates=["start_date", "end_date"])
campaigns     = pd.read_csv("Dataset/campaigns.csv")
redemptions   = pd.read_csv("Dataset/coupon_redemptions.csv", parse_dates=["redemption_date"])
coupons       = pd.read_csv("Dataset/coupons.csv")
demographics  = pd.read_csv("Dataset/demographics.csv")
# transactions is the biggest file (1.4 million rows), so it is stored compressed; read_csv opens .gz files directly
transactions  = pd.read_csv("Dataset/transactions.csv.gz", parse_dates=["transaction_timestamp"])

## 1.2 Structure of each table

In [3]:
# Put the tables in a dictionary so we can loop over them
tables = {
    "campaign_desc": campaign_desc,
    "campaigns": campaigns,
    "redemptions": redemptions,
    "coupons": coupons,
    "demographics": demographics,
    "transactions": transactions,
}

# For each table: number of rows and columns, missing values and duplicate rows
summary = []
for name, table in tables.items():
    summary.append({
        "table": name,
        "rows": table.shape[0],
        "columns": table.shape[1],
        "missing_values": table.isnull().sum().sum(),
        "duplicate_rows": table.duplicated().sum(),
    })
pd.DataFrame(summary)

,table,rows,columns,missing_values,duplicate_rows
0,campaign_desc,27,4,0,0
1,campaigns,6589,2,0,0
2,redemptions,2102,4,0,0
3,coupons,116204,3,0,4872
4,demographics,801,8,370,0
5,transactions,1469307,11,0,0


In [4]:
# Look at the first rows and the data types of the main tables
print(campaign_desc.head(), "\n")
print(campaign_desc.dtypes, "\n")
print(campaigns.head(), "\n")
print(redemptions.head(), "\n")
print(transactions.head(), "\n")
print(transactions.dtypes)

   campaign_id campaign_type start_date   end_date
0            1        Type B 2017-03-03 2017-04-09
1            2        Type B 2017-03-08 2017-04-09
2            3        Type C 2017-03-13 2017-05-08
3            4        Type B 2017-03-29 2017-04-30
4            5        Type B 2017-04-03 2017-05-07 

campaign_id               int64
campaign_type            object
start_date       datetime64[ns]
end_date         datetime64[ns]
dtype: object 

   campaign_id  household_id
0            1           105
1            1          1238
2            1          1258
3            1          1483
4            1          2200 

   household_id   coupon_upc  campaign_id redemption_date
0          1029  51380041013           26      2017-01-01
1          1029  51380041313           26      2017-01-01
2           165  53377610033           26      2017-01-03
3           712  51380041013           26      2017-01-07
4           712  54300016033           26      2017-01-07 

   household_id  store

**What we see:**

* `campaign_desc` has 27 campaigns with a type (A, B or C) and start and end dates.
* `campaigns` links households to campaigns, and `redemptions` records which of them redeemed a coupon.
* `transactions` has about 1.47 million product lines. Discounts are stored as positive amounts.
* `coupons` has some duplicate rows (the same coupon listed more than once). We only count its unique coupons, so duplicates do no harm.

## 1.3 Summary statistics

In [5]:
# Summary statistics of the numeric transaction columns
transactions[["quantity", "sales_value", "retail_disc", "coupon_disc", "coupon_match_disc"]].describe().round(2)

,quantity,sales_value,retail_disc,coupon_disc,coupon_match_disc
count,1469307.00,1469307.00,1469307.00,1469307.00,1469307.00
mean,104.08,3.13,0.54,0.02,0.00
std,1174.39,4.29,1.26,0.23,0.04
min,0.00,0.00,0.00,0.00,0.00
25%,1.00,1.29,0.00,0.00,0.00
50%,1.00,2.00,0.01,0.00,0.00
75%,1.00,3.49,0.68,0.00,0.00
max,89638.00,840.00,130.02,55.93,7.70


In [6]:
# Summary of the household demographics (all text columns, so we count the values)
for col in ["age", "income", "home_ownership", "marital_status", "household_size", "household_comp", "kids_count"]:
    print(demographics[col].value_counts(dropna=False), "\n")

age
45-54    288
35-44    194
25-34    142
65+       72
55-64     59
19-24     46
Name: count, dtype: int64 

income
50-74K       192
35-49K       172
75-99K        96
25-34K        77
15-24K        74
Under 15K     61
125-149K      38
100-124K      34
150-174K      30
250K+         11
175-199K      11
200-249K       5
Name: count, dtype: int64 

home_ownership
Homeowner             504
NaN                   233
Renter                 42
Probable Renter        11
Probable Homeowner     11
Name: count, dtype: int64 

marital_status
Married      340
Unmarried    324
NaN          137
Name: count, dtype: int64 

household_size
2     318
1     255
3     109
5+     66
4      53
Name: count, dtype: int64 

household_comp
2 Adults No Kids    258
1 Adult No Kids     255
2 Adults Kids       195
1 Adult Kids         93
Name: count, dtype: int64 

kids_count
0     513
1     159
3+     69
2      60
Name: count, dtype: int64 



In [7]:
# Campaigns by type, and how many households each campaign targeted
print(campaign_desc["campaign_type"].value_counts(), "\n")
print(campaigns.groupby("campaign_id").size().describe().round(1))

campaign_type
Type B    17
Type C     6
Type A     4
Name: count, dtype: int64 

count      27.0
mean      244.0
std       319.6
min        12.0
25%        65.0
50%       170.0
75%       219.0
max      1133.0
dtype: float64


## 1.4 Missing values

In [8]:
# Missing values per column in the demographics table
demographics.isnull().sum()

household_id        0
age                 0
income              0
home_ownership    233
marital_status    137
household_size      0
household_comp      0
kids_count          0
dtype: int64

In [9]:
#Demographics exist for only some households. How many of the targeted households have them?
targeted_households = campaigns["household_id"].unique()
has_demo = np.isin(targeted_households, demographics["household_id"])
print("Households targeted by at least one campaign:", len(targeted_households))
print("Of these, with demographics:", has_demo.sum(), f"({has_demo.mean():.0%})")

pairs_with_demo = campaigns["household_id"].isin(demographics["household_id"])
print(f"Household-campaign pairs without demographics: {(~pairs_with_demo).sum()} of {len(campaigns)} ({(~pairs_with_demo).mean():.0%})")

Households targeted by at least one campaign: 1559
Of these, with demographics: 756 (48%)
Household-campaign pairs without demographics: 2754 of 6589 (42%)


**Missing values:** the transactions, campaigns and redemptions tables have no empty cells. The gaps are in `demographics`: some fields are empty, and most targeted households are not in the table at all. We handle this in Step 2.